# Temperature Prediction Exploration

This notebook explains the same workflow used by the app, step by step.

Goal: use the previous few temperatures to predict the next temperature.

## 1. Import Libraries

We use:

- `pandas` to read the CSV file
- `numpy` to work with arrays
- `matplotlib` to draw charts
- `tensorflow.keras` to build the RNN

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Input, SimpleRNN
from tensorflow.keras.models import Sequential

ModuleNotFoundError: No module named 'matplotlib'

## 2. Load the Dataset

The dataset has two columns:

- `day`
- `temperature`

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "temperature.csv"

df = pd.read_csv(DATA_PATH)
df.head()

## 3. Visualize the Data

Before training a model, always look at the data. This helps you understand the pattern the model is trying to learn.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(df["day"], df["temperature"], marker="o")
plt.xlabel("Day")
plt.ylabel("Temperature")
plt.title("Temperature by Day")
plt.grid(True)
plt.show()

## 4. Create Sequences

An RNN learns from sequences.

With `sequence_length = 5`, one training example looks like this:

```text
[20, 21, 22, 23, 24] -> 25
```

The first 5 numbers are the input. The next number is the target.

In [ ]:
def create_sequences(data, sequence_length=5):
    inputs = []
    targets = []

    for start_index in range(len(data) - sequence_length):
        end_index = start_index + sequence_length
        inputs.append(data[start_index:end_index])
        targets.append(data[end_index])

    X = np.array(inputs, dtype=np.float32)
    y = np.array(targets, dtype=np.float32)

    # RNN input shape must be: examples, time steps, features
    X = X.reshape((X.shape[0], X.shape[1], 1))
    return X, y


temperatures = df["temperature"].to_numpy(dtype=np.float32)
X_raw, y_raw = create_sequences(temperatures, sequence_length=5)

print("X shape:", X_raw.shape)
print("y shape:", y_raw.shape)
print("First input sequence:")
print(X_raw[0].flatten())
print("First target:", y_raw[0])

## 5. Scale the Data

Neural networks usually train better when numbers are close to zero.

We scale temperatures using:

```text
scaled_value = (value - mean) / standard_deviation
```

Later, we convert predictions back to normal temperatures.

In [ ]:
mean = temperatures.mean()
std = temperatures.std()

scaled_temperatures = (temperatures - mean) / std
X, y = create_sequences(scaled_temperatures, sequence_length=5)

print("Mean:", mean)
print("Standard deviation:", std)
print("First scaled sequence:")
print(X[0].flatten())

## 6. Build the RNN Model

The model has:

- an input layer
- a `SimpleRNN` layer that reads the temperature sequence
- a `Dense` layer that outputs one predicted value

In [ ]:
sequence_length = 5

model = Sequential([
    Input(shape=(sequence_length, 1)),
    SimpleRNN(32),
    Dense(1),
])

model.compile(optimizer="adam", loss="mse")
model.summary()

## 7. Train the Model

`epochs` means how many times the model studies the training data.

`validation_split=0.2` means 20% of the examples are used to check how well the model is doing while training.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=30,
    restore_best_weights=True,
)

history = model.fit(
    X,
    y,
    epochs=100,
    batch_size=8,
    validation_split=0.2,
    shuffle=False,
    callbacks=[early_stopping],
    verbose=1,
)

## 8. Plot Training Loss

Loss is the model's error during training. Lower is better.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Progress")
plt.legend()
plt.grid(True)
plt.show()

## 9. Evaluate Predictions

The model predicts scaled values, so we convert predictions back to real temperatures.

In [ ]:
predictions_scaled = model.predict(X, verbose=0).flatten()

actual_temperatures = (y * std) + mean
predicted_temperatures = (predictions_scaled * std) + mean

mae = np.mean(np.abs(actual_temperatures - predicted_temperatures))
print(f"Mean absolute error: {mae:.2f}")

comparison = pd.DataFrame({
    "actual": actual_temperatures,
    "predicted": predicted_temperatures,
})
comparison.head(10)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(actual_temperatures, label="Actual")
plt.plot(predicted_temperatures, label="Predicted")
plt.xlabel("Sequence number")
plt.ylabel("Temperature")
plt.title("Actual vs Predicted Temperatures")
plt.legend()
plt.grid(True)
plt.show()

## 10. Predict the Next Temperature

Give the model 5 recent temperatures. It predicts the next one.

In [ ]:
new_sequence = np.array([45, 46, 47, 48, 49], dtype=np.float32)

scaled_sequence = (new_sequence - mean) / std
scaled_sequence = scaled_sequence.reshape(1, sequence_length, 1)

next_temperature_scaled = model.predict(scaled_sequence, verbose=0).flatten()[0]
next_temperature = (next_temperature_scaled * std) + mean

print("Input sequence:", new_sequence.astype(int).tolist())
print(f"Predicted next temperature: {next_temperature:.2f}")

## What This Notebook Teaches

You practiced the full machine learning workflow:

1. Load data
2. Visualize data
3. Convert data into sequences
4. Scale values
5. Build an RNN
6. Train the model
7. Evaluate predictions
8. Predict a new value

The Python scripts in `src/` use the same ideas, but are organized for reuse from the command line.